# Hands-on 7 — Make your own embeddings with Clay

**Chapter 5 · Patch embeddings and open models**

Every embedding so far was computed by someone else. Sometimes that is not enough: you need a date they did not process, a sensor they did not use, a region they skipped, or you want to fine-tune. Then you run a foundation model yourself. **Clay** (Clay Foundation, Apache-2.0) is the most accessible open EO foundation model: one checkpoint, any optical or radar sensor described by its band wavelengths, runs on a laptop CPU.

In this notebook you will:

1. load Clay v1.5 and understand what it needs as input (pixels, time, location, wavelengths, resolution),
2. fetch 12 Sentinel-2 chips over the study area straight from a public STAC catalogue,
3. produce one embedding per chip **and** one embedding per 80 m patch inside each chip,
4. check that Clay's embedding space agrees with AlphaEarth's,
5. know what it costs to scale this up, and when precomputed products are the better deal.

**Requirements.** First run downloads the 5.2 GB checkpoint (once, cached) and needs about 8 GB of free RAM while loading. Inference itself is light: ~0.3 s per chip on a laptop CPU. Run time about 5 minutes after the download.

In [ ]:
import math, time, datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch, yaml
import rasterio
from affine import Affine
from rasterio.crs import CRS
from rasterio.warp import transform as warp_xy
import pystac_client, odc.stac
from odc.geo.geobox import GeoBox
from huggingface_hub import hf_hub_download

import eoemb

torch.set_num_threads(max(1, torch.get_num_threads() - 1))
cache = eoemb.DATA_DIR / "cache"
AOI = (23.90, 38.00, 24.00, 38.10)

## 1. Load the model

Clay is distributed as a PyTorch Lightning checkpoint plus a small `metadata.yaml` that lists, for each supported sensor, the band names, their central wavelengths, and the mean and standard deviation used for normalisation. The wavelengths are how Clay handles *any* sensor: instead of learning "band 4", it learns "reflectance at 665 nm".

In [ ]:
from claymodel.module import ClayMAEModule

t0 = time.time()
ckpt = hf_hub_download("made-with-clay/Clay", "v1.5/clay-v1.5.ckpt", cache_dir=cache / "hf")
meta_path = cache / "clay" / "metadata.yaml"
if not meta_path.exists():
    import urllib.request
    meta_path.parent.mkdir(parents=True, exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/Clay-foundation/model/main/configs/metadata.yaml", meta_path)

module = ClayMAEModule.load_from_checkpoint(
    ckpt, model_size="large", metadata_path=str(meta_path),
    dolls=[16, 32, 64, 128, 256, 768, 1024], doll_weights=[1] * 7,
    mask_ratio=0.0, shuffle=False, map_location="cpu",
).eval()
n_params = sum(p.numel() for p in module.model.encoder.parameters())
print(f"Clay v1.5 encoder loaded in {time.time() - t0:.0f} s — {n_params / 1e6:.0f} M parameters")

meta = yaml.safe_load(open(meta_path))["sentinel-2-l2a"]
BANDS = list(meta["band_order"])
MEANS = np.array([meta["bands"]["mean"][b] for b in BANDS], dtype=np.float32)
STDS = np.array([meta["bands"]["std"][b] for b in BANDS], dtype=np.float32)
WAVES = [meta["bands"]["wavelength"][b] for b in BANDS]
pd.DataFrame({"band": BANDS, "wavelength_um": WAVES, "mean": MEANS, "std": STDS})

Ten Sentinel-2 bands, 10 m ground sampling distance (`gsd`). Clay wants chips of **256 × 256 pixels**, which at 10 m is 2.56 km. Inside a chip it works on 8 × 8-pixel patches (80 m), 1024 patches per chip, and adds one *class token* that summarises the whole chip.

## 2. Fetch Sentinel-2 chips from a STAC catalogue

We use the same cloud-free scene as the rest of the course (13 July 2024) from Element 84's Earth Search, a public STAC API over the Sentinel-2 COGs on AWS. No account needed. We cut a 4 × 3 grid of chips inside the AlphaEarth window (the window is 9.1 km wide, so three 2.56 km chips fit beside the 1 km margin).

In [ ]:
catalog = pystac_client.Client.open("https://earth-search.aws.element84.com/v1")
item = next(catalog.search(collections=["sentinel-2-l2a"], ids=["S2B_34SGH_20240713_0_L2A"]).items())
acq_time = dt.datetime.fromisoformat(item.properties["datetime"].replace("Z", "+00:00"))
print(item.id, acq_time, f"cloud cover {item.properties['eo:cloud_cover']:.1f}%")

z = np.load(cache / "aef_attica_2024.npz")
aef_tr = Affine.from_gdal(*z["transform"]); crs = CRS.from_string(str(z["crs"]))
aef = eoemb.dequantize(z["raw"])

CHIP = 256
origin_x = aef_tr.c + 100 * 10          # start 1 km inside the AlphaEarth window
origin_y = aef_tr.f - 100 * 10
NROWS, NCOLS = 4, 3
grid = [(origin_x + j * CHIP * 10, origin_y - i * CHIP * 10) for i in range(NROWS) for j in range(NCOLS)]

t0 = time.time()
chips, centres = [], []
for x0, y0 in grid:
    gbox = GeoBox((CHIP, CHIP), Affine(10, 0, x0, 0, -10, y0), "EPSG:32634")
    ds = odc.stac.load([item], bands=BANDS, geobox=gbox, chunks=None)
    chips.append(np.stack([ds[b].isel(time=0).values for b in BANDS]).astype(np.float32))
    lon, lat = warp_xy(crs, "EPSG:4326", [x0 + CHIP * 5], [y0 - CHIP * 5])
    centres.append((lat[0], lon[0]))
chips = np.stack(chips)                                  # (12, 10, 256, 256) reflectance × 10 000
print(f"{chips.shape} loaded in {time.time() - t0:.0f} s")

fig, axes = plt.subplots(NROWS, NCOLS, figsize=(7.5, 10))
for ax, c in zip(axes.ravel(), chips):
    ax.imshow(np.clip(np.moveaxis(c[[2, 1, 0]], 0, -1) / 3000, 0, 1) ** 0.8); ax.set_axis_off()
plt.suptitle("12 Sentinel-2 chips, 2.56 km each"); plt.tight_layout()

## 3. Build the input and run the encoder

Clay's input is a dictionary. Besides the normalised pixels it takes the acquisition **time** and **location**, both encoded as sine/cosine pairs so that December is close to January and 179° E is close to 179° W. These let the model condition on season and place.

In [ ]:
def encode_time(t: dt.datetime):
    week = t.isocalendar()[1] * 2 * math.pi / 52
    hour = t.hour * 2 * math.pi / 24
    return [math.sin(week), math.cos(week), math.sin(hour), math.cos(hour)]

def encode_latlon(lat, lon):
    lat, lon = math.radians(lat), math.radians(lon)
    return [math.sin(lat), math.cos(lat), math.sin(lon), math.cos(lon)]

def clay_embed(chips, times, latlons, batch_size=4):
    """Return (cls_embeddings (N, 1024), patch_embeddings (N, 32, 32, 1024)) for chips (N, 10, 256, 256)."""
    pixels = (chips - MEANS[None, :, None, None]) / STDS[None, :, None, None]
    cls_all, patch_all = [], []
    for s in range(0, len(chips), batch_size):
        batch = {
            "pixels": torch.tensor(pixels[s:s + batch_size], dtype=torch.float32),
            "time": torch.tensor([encode_time(t) for t in times[s:s + batch_size]], dtype=torch.float32),
            "latlon": torch.tensor([encode_latlon(*ll) for ll in latlons[s:s + batch_size]], dtype=torch.float32),
            "waves": torch.tensor(WAVES, dtype=torch.float32),
            "gsd": torch.tensor(float(meta["gsd"])),
        }
        with torch.no_grad():
            tokens, *_ = module.model.encoder(batch)         # (B, 1 + 32*32, 1024)
        cls_all.append(tokens[:, 0].numpy())
        patch_all.append(tokens[:, 1:].reshape(-1, 32, 32, 1024).numpy())
    return np.concatenate(cls_all), np.concatenate(patch_all)

t0 = time.time()
cls, patch_tokens = clay_embed(chips, [acq_time] * len(chips), centres)
print(f"{len(chips)} chips embedded in {time.time() - t0:.1f} s  →  cls {cls.shape}, patches {patch_tokens.shape}")

Two kinds of output:

- **class token** (1024 values per chip): a summary of the whole 2.56 km chip, comparable to a Major TOM patch embedding;
- **patch tokens** (32 × 32 × 1024 per chip): one embedding per 80 m patch, a small pixel-level product like AlphaEarth at 80 m.

Note the speed: a third of a second per chip on a CPU, after a one-off model load.

## 4. Look at the chip embeddings

In [ ]:
cls_n = cls / np.linalg.norm(cls, axis=1, keepdims=True)
sim = cls_n @ cls_n.T

fig, ax = plt.subplots(1, 2, figsize=(14, 6))
im = ax[0].imshow(sim, cmap="viridis"); plt.colorbar(im, ax=ax[0], shrink=0.8); ax[0].set_title("cosine similarity between the 12 chips (class token)")
ax[0].set_xticks(range(len(grid))); ax[0].set_yticks(range(len(grid)))
from sklearn.decomposition import PCA
pc = PCA(3).fit_transform(cls_n); lo, hi = pc.min(0), pc.max(0)
ax[1].imshow(np.clip((pc - lo) / (hi - lo), 0, 1).reshape(NROWS, NCOLS, 3)); ax[1].set_title("chips coloured by PCA of their embedding")
for k in range(len(grid)): ax[1].text(k % NCOLS, k // NCOLS, str(k), ha="center", va="center", color="w")
ax[1].set_axis_off(); plt.tight_layout()

Chips along the coast and in the town group together, the forest chips form another group. Compare the PCA colours with the true-colour mosaic above.

## 5. Inside a chip: the 80 m patch embeddings

Take one chip and colour its 32 × 32 patch tokens by PCA. This is a tiny embedding map at 80 m, produced from a single image.

In [ ]:
k = 5   # pick a mixed chip (settlement and fields, east column, second row)
pt = patch_tokens[k].reshape(-1, 1024)
pc = PCA(3).fit_transform(pt / np.linalg.norm(pt, axis=1, keepdims=True))
lo, hi = np.percentile(pc, [2, 98], axis=0)

fig, ax = plt.subplots(1, 3, figsize=(16, 5.5))
ax[0].imshow(np.clip(np.moveaxis(chips[k][[2, 1, 0]], 0, -1) / 3000, 0, 1) ** 0.8); ax[0].set_title(f"chip {k}, Sentinel-2 true colour")
ax[1].imshow(np.clip((pc - lo) / (hi - lo), 0, 1).reshape(32, 32, 3)); ax[1].set_title("Clay patch tokens, 80 m, PCA(3)")
x0, y0 = grid[k]
r0, c0 = eoemb.lonlat_to_rowcol(*warp_xy(crs, "EPSG:4326", [x0], [y0]), aef_tr, crs)
ax[2].imshow(eoemb.pca_rgb(aef[:, r0[0]:r0[0] + CHIP, c0[0]:c0[0] + CHIP])); ax[2].set_title("AlphaEarth 2024 for the same chip, 10 m, PCA(3)")
for a in ax: a.set_axis_off()
plt.tight_layout()

Same structures, different scales. Clay's tokens, from one July image and 8-pixel patches, separate the settlement, the fields and the scrub in broad strokes, but they are noisy and speckled; AlphaEarth, with a year of multi-sensor context and 10 m detail, draws the same structures cleanly. **This is the trade you make when you run a model yourself: full control over date and sensor, in exchange for the depth that a multi-temporal, multi-sensor product has baked in.**

## 6. Does Clay agree with AlphaEarth?

Average the AlphaEarth pixels under each chip to get one vector per chip, then compare the two 12 × 12 similarity matrices, as in Hands-on 2.

In [ ]:
from scipy.stats import spearmanr

aef_chip = []
for x0, y0 in grid:
    r0, c0 = eoemb.lonlat_to_rowcol(*warp_xy(crs, "EPSG:4326", [x0], [y0]), aef_tr, crs)
    block = aef[:, r0[0]:r0[0] + CHIP, c0[0]:c0[0] + CHIP].reshape(64, -1)
    v = np.nanmean(block, axis=1); aef_chip.append(v / np.linalg.norm(v))
aef_chip = np.array(aef_chip)
sim_aef = aef_chip @ aef_chip.T

iu = np.triu_indices(len(grid), k=1)
rho = spearmanr(sim[iu], sim_aef[iu]).statistic
print(f"Spearman correlation between Clay and AlphaEarth chip similarities: {rho:.2f}")
plt.figure(figsize=(5, 5)); plt.scatter(sim_aef[iu], sim[iu], s=18)
plt.xlabel("AlphaEarth similarity (mean of pixels)"); plt.ylabel("Clay similarity (class token)"); plt.title(f"{len(iu[0])} chip pairs, ρ = {rho:.2f}");

## 7. What it costs to scale

| | this notebook | Attica (3 800 km²) | Greece (132 000 km²) |
|---|---|---|---|
| chips (2.56 km) | 12 | ~600 | ~20 000 |
| CPU time at 0.3 s/chip | 4 s | 3 min | 2 h |
| Sentinel-2 to download (10 bands) | 15 MB | 0.8 GB | 26 GB |
| for 12 dates a year | | 10 GB, 36 min | 300 GB, 24 h |

One date over a country is a lunch break on a laptop. A dense time series over a country is cloud territory: a GPU does 50–100 chips per second, and processing *next to the data* (Copernicus Data Space Ecosystem, CREODIAS, AWS in the Sentinel-2 region) avoids the download entirely. That is the niche of services like CloudFerro's GeoAI embedding generation, and it is why precomputed global products exist: someone has already paid this bill once, for everyone.

**Run it yourself when** you need a specific date or sensor, want patch-level control, or plan to fine-tune. **Use a precomputed product when** annual (or the provider's cadence) is fine and the sensors match your needs. Most projects start with the second and graduate to the first only where it earns its keep.

---
## Exercises

**E1.** Find a cloud-free Sentinel-2 scene of the same area from **February 2024** (`catalog.search(..., datetime="2024-02-01/2024-02-29", query={"eo:cloud_cover": {"lt": 10}})`) and embed the same 12 chips. How similar is each chip to its July self? Which chips change most between seasons, and does that match what you expect from farmland vs forest vs town?

**E2.** Instead of the class token, use the **mean of the 1024 patch tokens** as the chip embedding. Recompute the agreement with AlphaEarth. Which summary agrees better?

**E3.** Use the patch tokens of all 12 chips as an 80 m embedding layer: label each patch with the majority WorldCover class under it (`data/worldcover_2021_attica.tif`), train a logistic regression on 6 chips and test on the other 6. Compare with the same experiment on AlphaEarth averaged to 80 m.